# Deep-dive into a Ray's task and object lifecycle

In this notebook, we will provide a deep-dive into a Ray task and object lifecycle. 

<div class="alert alert-info">
    
__Here is the roadmap for this notebook:__

1. High-Level Overview of a Task's Lifecycle
2. The Main Components of a Ray Cluster
3. Task Execution Stages in Detail:
    1. Mapping Execution Stages to Cluster Components
    2. Task Submission in Detail
    4. Autoscaling in Detail
    5. Task Scheduling in Detail
    6. Result Handling in Detail
4. Object Management in Ray
   1. Ray's memory model
   2. Object Failure handling
   3. Object Store spilling
5. Overview of Scheduling Strategies
    1. How does a raylet classify nodes?
    2. Default Scheduling Strategy
    3. Node Affinity Strategy
    4. SPREAD Scheduling Strategy
    5. Placement Group Scheduling Strategy
</div>

Note in most cases, the notebook applies to Java, C++, and Python tasks. However certain remarks mainly focus on peculiarities of python tasks.

# High-level overview of a task's lifecycle

We start by visualizing a task's execution using the following diagram:

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/task_execution_detail_0_v3.svg.svg" width="1000px">

In case you skipped it, this same diagram was presented in the high-level overview notebook of Ray tasks.

We will proceed to add more color to this diagram providing useful details for each step of the process

# The main components of a Ray cluster

A Ray cluster consists of:
- One or more **worker nodes**, where each worker node consists of the following processes:
    - **worker processes** responsible for task submission and execution.
    - A **raylet** responsible for resource management and task placement.
- One of the worker nodes is designated a **head node** and is responsible for running 
  - A **global control service** responsible for keeping track of the **cluster-level state** that is not supposed to change too frequently.
  - An **autoscaler** service responsible for allocating and removing worker nodes by integrating with different infrastructure providers (e.g. AWS, GCP, ...) to match the resource requirements of the cluster.

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/ray_cluster_detail_0_v3.svg" width="1000px">

Based on [Ray V2 whitepaper -> Architecture Overview -> Design -> Components](https://docs.google.com/document/d/1tBw9A4j62ruI5omIJbMxly-la5w4q_TjyJgJL_jN2fI/preview#heading=h.cclei73t0j5p)

Note that the color coding is intentional and consistent across all diagrams in this notebook.

## Driver and Job Submission

One of the worker processes is designated as the **driver process**. By default, the driver runs on the head node. 

The driver is responsible for:
- executes the top-level application (e.g. __main__ in Python script)
- calls `ray.init()` to connect to an existing Ray cluster or to start a new one
- submits tasks and actors
- unlike other worker processes, the driver process does not execute tasks

As for the job:
- it is the collection of tasks and actors originating recursively from the same driver process
- there is a 1:1 mapping between a driver process and a job


# Task execution stages in detail

## Mapping execution stages to cluster components

Now that we are familiar with the different components on a Ray cluster, here is our same task execution diagram revisited with colors indicating which component is responsible for each step.

- One **worker process** submits the task
- The cluster **autoscaler** will handle upscaling nodes to meet new resource requirements
- **Raylet(s)** will handle task scheduling/placement on a worker
- **One worker process** executes the task
- The result information is sent back to the **submitter worker** once complete

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/task_execution_detail_1_v3.svg.svg" width="1000px">

Note: the autoscaler scaling step is entirely optional and depends on the availability of resources on the cluster. (i.e. it is triggered after a Raylet fails to find a node to schedule the task on)

## Task submission in detail

### Exporting and loading function code

Remember a task wraps around a given function. The worker executing a task, is executing its underlying function.

Here are the steps that Ray follows to export and load a task's function:

1. The submitter worker will serialize a task's function definition
    - In the case of Python, Ray makes use of a variant of pickle (cloudpickle) to serialize the function
2. The submitter worker will export the function definition to the GCS Store
3. The executor worker will load and cache the function definition from the GCS Store
4. The executor worker will deserialize the code and execute the function

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/task_execution_detail_export_load_v3.svg" width="1200px">

<details>
<summary>Click to see the implementation details:</summary>

- Exporting Function to GCS Store
    1. [Python .remote calls ._remote](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/remote_function.py#L140)
    2. [Python ._remote pickles the function](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/remote_function.py#L299)
    3. [Python ._remote call exports the function via the function manager.export](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_private/function_manager.py#L273)
    4. [Which calls the cython GcsClient.internal_kv_put](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L2579)
    5. [Which calls the gcs_client.cc PythonGcsClient::InternalKVPut](https://github.com/ray-project/ray/blob/55ab6dfd6b415f8795dd1dfed7b3fde2558efc46/src/ray/gcs/gcs_client/gcs_client.cc#L312) that sets the key, value in the proper namespace 
 - Importing Function from GCS Store
    1. [When instantiating a CoreWorker, we add task receivers which will callback CoreWorker::ExecuteTask](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/core_worker.cc#L147)
    2. [CoreWorker::ExecuteTask() will prepare a RayFunction and submit it to its execution callback](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/core_worker.cc#L2721C21-L2721C44)
    3. [The task execution callback in the case of python will execute the function from cython given the set task_execution_handler](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L3075C43-L3075C65)
    4. [The task execution handler will execute the task with a cancellation handler](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L2064C17-L2064C55)
    5. [The handler will call the function_manager.get_execution_info](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L1944)
    6. [function_manager.get_execution_info will in turn call function_manager._wait_for_function](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_private/function_manager.py#L393)
    7. [function_manager._wait_for_function will in turn call function_manager.fetch_and_register_remote_function](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_private/function_manager.py#L455C33-L455C67)
    8. [function_manager.fetch_and_register_remote_function will in turn call function_manager.fetch_registered_method](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_private/function_manager.py#L299C37-L299C60)
    9. [function_manager.fetch_registered_method will in turn call gcs_client.internal_kv_get to read the function defintion from the GCS KV store](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_private/function_manager.py#L281)
 - Caching the function definition:
   1. [As a continuation the execution_infos in-memory dictionary mapping is updated to store the function definition](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L1946)

</details>

#### Related anti-pattern: Redefining the same remote function or class harms performance

Decorating the same function or class multiple times using the `ray.remote` decorator leads to slow performance in Ray. 

For each Ray remote function or class, Ray will pickle it and upload to GCS. Later on, the worker that runs the task or actor will download and unpickle it. 

Each decoration of the same function or class generates a new remote function or class from Ray’s perspective. As a result, the pickle, upload, download and unpickle work will happen every time we redefine and run the remote function or class.

##### Anti-pattern:
```python
outputs = []
for i in range(10):

    @ray.remote
    def double(i):
        return i * 2

    outputs.append(double.remote(i))
outputs = ray.get(outputs)
# The double remote function is pickled and uploaded 10 times.
```

##### Improved approach:
```python
@ray.remote
def double(i):
    return i * 2


outputs = []
for i in range(10):
    outputs.append(double.remote(i))
outputs = ray.get(outputs)
# The double remote function is pickled and uploaded 1 time.
```

We should define the same remote function or class outside of the loop instead of multiple times inside a loop so that it’s pickled and uploaded only once.



### Resolving dependencies and data locality

Here are some key steps in task submission:

1. A submitter worker will resolve a task's dependency locations before creating and submitting the task.
2. A submitter worker will choose the worker node that has most of the dependency data local to it.
3. A submitter worker will request what Ray calls a "Worker Lease" from the raylet on the chosen data-locality-optimal node

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/task_execution_detail_resolving_deps_data_locality_v3.svg" width="1200px">


let's unpack the above steps

#### Resolving dependencies in detail

Given a particular task `task1` that depends on, objects `A` and `B` as inputs

The submitter worker process will perform these two main steps

1. Wait for each object to be available via async callbacks
    - remember `A` and `B` could very well be the outputs of a different task, hence why we need to wait 
    - as long as `A` and `B` are available somewhere in the cluster, they are considered resolved - no need to be local to the submitter worker
2. Proceed with scheduling now that all dependencies are resolved

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/resolving_deps_v3.svg" width="800px">

Note: Later in the notebook we will discuss ray's distributed ownership and object store which will clarify *how* the worker can check if the objects are ready. 

<details>
<summary>Click to see the implementation details:</summary>

- Resolving dependency metadata:
    1. [Python .remote calls ._remote](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/remote_function.py#L140)
    2. [python ._remote call calls submit_task](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/remote_function.py#L420)
    3. [submit_task calls the cython submit_task function defined here](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L3574)
    4. [cython submit_task Delegates to C++ CoreWorker::SubmitTask](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L3643)
    5. [CoreWorker::SubmitTask calls direct_task_submitter.SubmitTask](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/core_worker.cc#L1949)
    6. [direct_task_submitter.SubmitTask calls ResolveDependencies to resolve dependencies](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/transport/direct_task_transport.cc#L28)
    7. [ResolveDependencies calls InlineDependencies](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/transport/dependency_resolver.cc#L117)
    8. [InlineDependencies fetches task metadata like the size](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/transport/dependency_resolver.cc#L44C10-L44C10)
</details>

#### Data locality resolution in detail

The submitter process will prefer the raylet on the node that has the **most number of object argument bytes** already local.

The diagram shows the same particular task `task1` we saw before. 

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/data_locality_v3.svg" width="800px">

Note: Small caveat: "enforcing data locality" stage is skipped in case the task's specified scheduling policy is stringent (e.g. a node-affinity policy). Scheduling policies will be discussed in more detail later in the notebook.



<details>
<summary>Click to see the implementation details:</summary>

- Finding the data-locality-optimal node:
    1. [Python .remote calls ._remote](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/remote_function.py#L140)
    2. [python ._remote call calls submit_task](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/remote_function.py#L420)
    3. [submit_task calls the cython submit_task function](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L3643)
    4. [cython submit_task Delegates to SubmitTask from c++ Core Worker with calls direct_task_submitter.SubmitTask](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/core_worker.cc#L1949)
    5. [direct_task_submitter.SubmitTask calls ResolveDependencies to resolve dependencies](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/transport/direct_task_transport.cc#L28)
    6. [direct_task_submitter.SubmitTask as a callback will now call RequestNewWorkerIfNeeded](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/transport/direct_task_transport.cc#L135)
    7. [RequestNewWorkerIfNeeded will in turn call GetBestNodeForTask](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/transport/direct_task_transport.cc#L394)
    8. [GetBestNodeForTask will pick a node for locality in case the scheduling strategy is not stringent (i.e. node affinity or spread)](https://github.com/ray-project/ray/blob/master/src/ray/core_worker/lease_policy.cc#L39)
    9. [GetBestNodeIdForTask will find the node with the most object bytes](https://github.com/ray-project/ray/blob/master/src/ray/core_worker/lease_policy.cc#L47C1-L48C1)

</details>

## Task scheduling in detail

Now that a worker lease request is sent, here are the steps that follow to schedule a task

- The **raylet on the data-locality-optimal node**:
    - Receives the worker lease request 
    - Receives a view of the entire cluster state from the GCS via a periodic broadcast
    - Makes a decision: which node is the best to schedule the task on
- The **raylet on the best node** now:
    - Attempts to reserve the resources on the node to satisfy the lease
    - Updates the GCS in case it succeeds to reserve the resources via a periodic message
 
This is shown in the below diagram, the potential autoscaling step prior to finding a best node is left out to simplify


<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/task_execution_detail_find_best_node_allocate_resources_v3.svg" width="1200px">



<details>
<summary>Click to see the implementation details:</summary>

- Raylet Receives Worker Lease Request:
    1. [When a worker lease request comes in a raylet's NodeManager::HandleRequestWorkerLease gets called](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/node_manager.cc#L1783)
    2. [NodeManager::HandleRequestWorkerLease will delegate a call to ClusterTaskManager::QueueAndScheduleTask()](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/node_manager.cc#L1830)
    3. [ClusterTaskManager::QueueAndScheduleTask it will delegate a call to ClusterTaskManager::ScheduleAndDispatchTasks()](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/cluster_task_manager.cc#L64)
    4. [ClusterTaskManager.ScheduleAndDispatchTasks() will call ClusterTaskManager.ScheduleOnNode()](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/cluster_task_manager.cc#L192)
- Best Scheduleable Node is a different Node
    1. [Check that indeed we can allocate resources on the remote node by calling ClusterResourceManager.AllocateRemoteTaskResources](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/cluster_task_manager.cc#L440)
    2. [Send a redirection back to the task submitter by sending a reply callback with the remote raylet address calling send_reply_callback()](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/cluster_task_manager.cc#L456)
    3. [The task submitter CoreWorkerDirectTaskSubmitter::SubmitTask will now handle the callback by sending the worker lease request to the remote raylet](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/transport/direct_task_transport.cc#L505C1-L506C1) 
- Best Scheduleable Node is the same as the Local (data-locality-optimal) Node
    1. [If the node id is the same as the local node id, then LocalTaskManager.QueueAndScheduleTask() is called](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/cluster_task_manager.cc#L421)
    2. [LocalTaskManager.QueueAndScheduleTask will first wait for the task arguments to be ready calling LocalTaskManager.WaitForTaskArgsRequests](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/local_task_manager.cc#L66)
    3. [LocalTaskManager.ScheduleAndDispatchTasks() will in turn call LocalTaskManager.DispatchScheduledTasksToWorkers()](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/local_task_manager.cc#L96)
    4. [LocalTaskManager.DispatchScheduledTasksToWorkers() will in turn pop a worker from the WorkerPool.PopWorker](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/local_task_manager.cc#L267)
- Allocating a worker and starting runtime environment
  - [Calling WorkerPool.PopWorker to assing a worker to the task by popping a worker from the pool](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/worker_pool.cc#L1150) will perform the following :
      - Worker Allocation: [Try to reuse existing idle workers that match the task's requirements. If no suitable worker is available, it starts a new one.](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/worker_pool.cc#L1250)
      - Runtime Environment Management: [Handles the creation or retrieval of a runtime environment necessary for the task](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/worker_pool.cc#L1261)

</details>

### Scheduling hotpath: leveraging leases and caches

- A scheduling request at task submission can reuse a leased worker if it has the same:
    - Resource requirements as these must be acquired from the node during task execution.
    - Shared-memory task arguments, as these must be made local on the node before task execution.
    - Runtime Environment: The leased worker must be started within the same runtime environment.

Leases optimize scheduling by reducing the need for communication with the scheduler for similar requests.

- This "hot path" most commonly occurs for **subsequent task executions**. We visualize it in the diagram below. Note how we skip:
    - sending a request to a raylet altogether
    - storing and fetching the function code in GCS

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/task_execution_scheduling_hot_path_v3.svg" width="1200px">


Note: the lease remains active as long as the caller and leased worker are alive, and the raylet ensures that no other client may use the worker while the lease is active. To ensure fairness, a caller returns the idle worker if no more task remains or if enough time has passed (e.g., a few hundred milliseconds).

## Autoscaling in detail

- **Question**: What happens if a raylet fails to find a "best node for a task"? Imagine a task that is requesting GPU resources when all the running worker nodes are CPU only.
- **Answer**: The task gets stuck in a pending state until the autoscaler adds GPU nodes to the cluster.

More specifically, here is how autoscaling works:

- The worker process submits tasks which request resources such as GPU.
- The raylet attempts to find the best node for the task.
- The raylet fails to find a node that satisfies the task requirements
- The raylet will put the task in a pending state
- The GCS will periodically poll resource usage and receive resource updates from all the raylets
- The autoscaler will:
    - periodically fetch the snapshots from GCS.
    - look at the resources available in the cluster, resources requested, what is pending and calculates the number of nodes to satisfy both running and pending tasks.
    - run a bin-packing algorithm to calculate the number of nodes to satisfy both running and pending tasks.
    - add or remove nodes from the cluster via the node provider interface (e.g. AWS interface)
- When the node comes up it registers with the cluster and accepts application workload.


<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/task_execution_autoscaling_v2.svg" width="1200px">

<details>
<summary>Click to see the implementation details:</summary>

- GCS polling resource usage:
    - [GcsServer::DoStart calls GcsServer::InitGcsResourceManager](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/gcs/gcs_server/gcs_server.cc#L195)
    - [GcsServer::InitGcsResourceManager periodically pulls resource load data from each node by calling GetResourceLoad](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/gcs/gcs_server/gcs_server.cc#L374C1-L375C1)
    - [NodeManager::HandleGetResourceLoad responds to requests for resource load data from individual nodes by calling ClusterTaskManager::FillResourceUsage](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/node_manager.cc#L1763)
    - [ClusterTaskManager::FillResourceUsage in turn calls SchedulerResourceReporter::FillResourceUsage which populates load information and PopulateResourceViewSyncMessage which poipulates usage information](https://github.com/ray-project/ray/blob/master/src/ray/raylet/scheduling/cluster_task_manager.cc#L351C1-L352C1)    

- Raylet sending resource updates:
    - [GcsServer::DoStart calls GcsServer::InitGcsResourceManager](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/gcs/gcs_server/gcs_server.cc#L195)
    - [The GCS resource manager](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/gcs/gcs_server/gcs_server.cc#L341) and raylet share the the cluster resource manager
    - [The cluster resource manager will periodically register a call from the raylet AddorUpdateNode which is called only after the resource is modified for a given time delta GetNodeResourceModifiedTs](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/cluster_resource_manager.cc#L34)
    - [AddorUpdateNode will call Node(node_resources) updating the cluster resource manager's view of a given node's resources](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/cluster_resource_manager.cc#L70)

- Autoscaler logic
    - [InitMonitorServer() in GcsServer::DoStart creates a GcsMonitorServer a shim responsible for providing a compatible interface between GCS and `monitor.py`](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/gcs/gcs_server/gcs_server.cc#L231)
    - [python/ray/autoscaler/_private/monitor.py -> Monitor: periodically collect stats from GCS and trigger autoscaler update](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/autoscaler/_private/monitor.py#L126)
    - [python/ray/autoscaler/_private/resource_demand_scheduler.py contains core autoscaling logic](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/autoscaler/_private/resource_demand_scheduler.py#L102C7-L102C30)
    - [python/ray/autoscaler/node_provider.py provides node provider interface](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/autoscaler/node_provider.py#L13)

</details>

## Result handling in detail

### Cluster components zoom-in

Let's revisit our mental model for the Ray cluster and add some more detail to which components control and manage objects in Ray.

- Each worker process stores:
    - **An ownership table** contains system metadata (object sizes, locations and reference counts) for the objects to which the worker has a reference
    - **An in-process store** used to store small objects.
- Each raylet runs:
    - A **shared-memory object store** responsible for storing, transferring, and spilling large objects. The individual object stores in a cluster comprise the _Ray distributed object store_

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/ray_cluster_distributed_ownership_v3.svg" width="1200px">

<!-- 
Reference: 
- See [V2 architecture document -> Architecture Overview -> Design -> Components](https://docs.google.com/document/d/1tBw9A4j62ruI5omIJbMxly-la5w4q_TjyJgJL_jN2fI/preview#heading=h.cclei73t0j5p)
 -->

### Object handling post execution

Let's take a look at the steps involved in object handling:

- The submitter worker creates an object reference for the output of the task in its ownership table
- The submitter worker then submits the task for scheduling
- The executor worker will execute the task function
- The executor worker will then prepare the return object
    - If the return object is small <100KB:
        - Return the values inline directly to the submitter's in-process object store.
    - If the return object is large:
        - Store the objects in the raylet object store
- The executor updates the submitter's ownership table with the location of the object

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/task_execution_detail_distributed_object_store_ownership_v4.svg" width="1200px">

<details>
<summary>Click to see the implementation details:</summary>

- Object handling post execution:
    1. [When instantiating a CoreWorker, we add task receivers which will callback CoreWorker::ExecuteTask](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/core_worker.cc#L147)
    2. [CoreWorker::ExecuteTask() will prepare a RayFunction and submit it to its execution callback](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/core_worker/core_worker.cc#L2721C21-L2721C44)
    3. [The task execution callback in the case of python will execute the function from cython given the set task_execution_handler](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L3075C43-L3075C65)
    4. [The task execution handler will execute the task with a cancellation handler](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L2064C17-L2064C55)
    5. [The handler will call execute_task handling a KeyboardInterrupt error](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L1960C7-L1960C7)
    6. [execute_task will invoke the function_executor](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L1675)
    7. [execute_task will store the outputs in the object store](https://github.com/ray-project/ray/blob/releases/2.8.1/python/ray/_raylet.pyx#L1810C12-L1810C12) 

</details>

### Distributed ownership work in Ray

#### How does it work?
The process that submits a task is considered to be the owner of the result of the task

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/distributed_ownership_overview_v4.svg" width="1000px">

#### Upsides to distributed ownership

- Latency: Faster than communicating all ownership information back to a head node.
- Scalability: There is no central bottleneck when attempting to scale the cluster given every worker maintains its own ownership information.

#### Downsides to distributed ownership

- objects fate-share with their owner
    - i.e. even though the object is available on an object store in node 2, if node 1 fails, the owner fails, and the object is no longer reachable

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/distributed_ownership_fate_share_with_owner_v4.svg" width="1200px">

To see this in action, let's look at the following example:

In [ ]:
!python distributed_ownership_fate_share_with_owner.py

### Distributed object store

The raylet's object store can be thought of as shared memory across all workers on a node.

For values that can be zero-copy deserialized, passing the ObjectRef to `ray.get` or as a task argument will return a direct pointer to the shared memory buffer to the worker.

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/distributed_ownership_data_sharing_v4.svg" width="800px">


#### Downsides to a shared object-store

Sharing the object store also means that "worker" processes fate-share with their local raylet process.

A simple mental model to have is `raylet = node` if a raylet fails, all workloads on node will fail 

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/distributed_ownership_fate_share_with_raylet_v4.svg" width="800px">

# Object Lifecycle in Ray

## Ray memory model

Ray manages memory in several ways to efficiently handle distributed tasks:

1. **Heap memory**:
   - Used by workers to execute tasks and actors.
   - Used to store small objects (less than 100KB) and Ray metadata.
   - High memory pressure can cause Ray to terminate some tasks to free up resources.

2. **Shared memory**:
   - Serves as the medium for passing data between tasks.
   - Large objects (greater than 100KB) are stored in a shared memory space, using up to 30% of a node's memory.
   - If more space is needed, objects can be spilled to disk or stored on disk in a slower-access format.

### Design Tradeoffs: Choosing where to store objects

The following table outlines the tradeoffs between storing objects in the heap memory (in-process store) versus the shared memory (distributed object store):

| **Design Aspect**           | **In-Process Store**                                      | **Distributed Object Store**                          |
|-----------------------------|----------------------------------------------------------|------------------------------------------------------|
| **Resolution Time**         | Fast (direct memory copy)                                | Slower (requires RPCs)                              |
| **Memory Footprint**        | Higher (multiple copies across processes)               | Lower (shared memory reduces duplication)           |
| **Throughput**              | Limited by owner process                                | Scales with number of nodes                         |
| **Data Sharing**            | Copies needed for multiple processes                    | Shared memory allows multiple processes to access   |
| **Object Size Constraints** | Limited by machine’s memory capacity                    | Can reference objects larger than a single machine's memory |


This setup allows Ray to optimize performance and resource usage in distributed applications.

Here is a simplified diagram of the Ray memory model:

<img src="https://docs.ray.io/en/latest/_images/memory.svg" width="600">

### Example of a numpy array in the object store

We will show two tasks:
- `producer_task`: creates a numpy array of size 4 GiB
- `consumer_task`: reads the output of `producer_task`

Here is the code
```python
@ray.remote
def producer_task(size_mb: int = 4 * 1024) -> np.ndarray:
    array = np.random.rand((1024**2 * size_mb // 8)).astype(np.float64)
    return array


@ray.remote
def consumer_task(array: np.ndarray) -> None:
    assert isinstance(array, np.ndarray)
    assert not array.flags.owndata

arr_ref = producer_task.remote()  # Produce a 4 GiB array.
output_ref = consumer_task.remote(arr_ref)  # Consume the array.
```

What happens under the hood?
- `producer_task` will:
    - allocate memory on the worker heap to create the array
    - allocate memory on the object store to store the array effectively calling `ray.put`
- `consumer_task` will:
    - directly access the array in the object store (zero-copy deserialization)
    - only incur the cost of copying the array if it is running on a different node.

Here is the diagram showing how the object store is used:

<img src="https://anyscale-materials.s3.us-west-2.amazonaws.com/ray-data-deep-dive/producer-consumer-object-store-v2.png" width="600">

Let's run a script that will inspect the memory usage of the tasks.

In [ ]:
!python memory_inspection.py

## Memory management

### Primary vs secondary copies

- For small values, the **executing worker** sends the result directly to the owner/submitter worker. The owner then stores it in its in-process memory. This value remains available until all references to it are gone, at which point it is deleted.
- For large values, the **executing worker** stores the result in its local shared memory. This initial copy, known as the **primary copy**, is special because it is **never evicted as long as there is an active reference**. The Raylet ensures this by pinning the primary copy—keeping a reference to the memory buffer where it’s stored. This prevents the object store from removing it.

See the diagram below for a visual representation of the primary and secondary copies:
<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/primary-vs-secondary-copies.png" width="1000">


## Object Failure

In the event of a system failure, Ray will attempt to recover any lost objects. If recovery is not possible, an application-level exception will be thrown when a worker tries to access the object's value.

### Recovery Guarantees

- **Owner Alive**: If the owner of an object is still alive, Ray will attempt to recover the object. If recovery fails, the owner will provide an exception with the cause.
- **Owner Dead**: If the owner has died, any worker trying to access the object will receive a generic error about the owner’s death, even if object copies still exist in the cluster.

### Small Objects

- Small objects are stored in the owner’s in-process object store. If the owner dies, these objects are lost. Workers attempting to access these objects will receive an error indicating the owner’s death.

### Large Objects and Lineage Reconstruction

- **Lost from Distributed Memory**: Non-primary copies of an object can be lost without issue. If the primary copy is lost, the owner will try to designate a new primary copy using the object directory.
- **Object Reconstruction**: If no other copies exist, Ray will attempt to reconstruct the object by re-executing the task that created it. If task dependencies are also lost, they will be recursively reconstructed.

### Lineage Reconstruction

- **Lineage Ref Count**: Each object has a lineage ref count, indicating the number of tasks that depend on it and can be re-executed. Once this count reaches zero, Ray will garbage-collect the task specification.
- **Memory Usage**: Cached lineage can increase driver memory usage. Ray workers will evict cached lineage if it exceeds a system-wide threshold (default 1GB).

### Limitations of Lineage Reconstruction

- Objects and their dependencies must be generated by a task. Objects created by `ray.put` are not recoverable.
- Tasks must be deterministic and idempotent.
- Tasks can be re-executed up to their maximum retries. By default, non-actor tasks can be retried up to 3 times, adjustable with the `max_retries` parameter.

See the below test inspired from the [Ray test suite](https://github.com/ray-project/ray/blob/a04cb06bb1a2c09e93b882b611492d62b8d1837a/python/ray/tests/test_reconstruction.py#L126) for an example of lineage reconstruction:

```python
@pytest.mark.parametrize("reconstruction_enabled", [False, True])
def test_basic_reconstruction(config, ray_start_cluster, reconstruction_enabled):
    cluster = ray_start_cluster
    # Head node with no resources.
    cluster.add_node(
        num_cpus=0,
        _system_config=config,
        enable_object_reconstruction=reconstruction_enabled,
    )
    ray.init(address=cluster.address)
    # Node to place the initial object.
    node_to_kill = cluster.add_node(
        num_cpus=1, resources={"node1": 1}, object_store_memory=10**8
    )
    cluster.wait_for_nodes()

    @ray.remote(max_retries=1 if reconstruction_enabled else 0)
    def create_large_object():
        return np.zeros(10**7, dtype=np.uint8)

    @ray.remote
    def process_large_object(x):
        return

    # Create a large object on node1
    obj = create_large_object.options(resources={"node1": 1}).remote()
    # Create a dependent task that will use the large object
    ray.get(process_large_object.options(resources={"node1": 1}).remote(obj))
    # Remove the node that has the large object
    cluster.remove_node(node_to_kill, allow_graceful=False)
    # Add a new node with the same resource config
    node_to_kill = cluster.add_node(
        num_cpus=1, resources={"node1": 1}, object_store_memory=10**8
    )

    if reconstruction_enabled:
        # If the large object is lost and reconstruction is enabled, the following call will succeed
        # i.e create_large_object will get called again and the object will be reconstructed
        # then process_large_object can run successfully
        ray.get(process_large_object.remote(obj))
    else:
        # Both the dependent task and the object will be lost
        with pytest.raises(ray.exceptions.RayTaskError):
            ray.get(process_large_object.remote(obj))
        with pytest.raises(ray.exceptions.ObjectLostError):
            ray.get(obj)

    # Losing the object a second time will cause reconstruction to fail because
    # we have reached the max task retries.
    cluster.remove_node(node_to_kill, allow_graceful=False)
    cluster.add_node(num_cpus=1, resources={"node1": 1}, object_store_memory=10**8)

    if reconstruction_enabled:
        with pytest.raises(
            ray.exceptions.ObjectReconstructionFailedMaxAttemptsExceededError
        ):
            ray.get(obj)
    else:
        with pytest.raises(ray.exceptions.ObjectLostError):
            ray.get(obj)
```

## Object Spilling

Ray supports spilling objects to external storage when the object store reaches capacity. This feature enables out-of-core data processing and supports memory-intensive distributed applications.

### External Storage Options
Ray provides a pluggable interface for external storage, with two default options:
- **Local Storage (Stable):** Uses local disk by default, requiring no additional configuration.
- **Distributed Storage (Experimental):** Currently supports Amazon S3, offering better fault tolerance at the cost of slower access speeds.

### Components of the Object Spilling Protocol
1. **Local Object Manager:** Tracks object metadata and coordinates IO workers and communication with other raylets.
2. **Shared-Memory Object Store**
3. **IO Workers:** Python processes responsible for spilling and restoring objects.
4. **External Storage:** Stores objects that exceed the object store's memory capacity.

### Spilling Process
When memory is insufficient to create new objects, Ray initiates object spilling. Only the primary copy of an object is spilled, ensuring a single spilled copy per object in the cluster. Non-primary copies can be evicted immediately.

**Spilling Protocol Steps:**
- The local object manager identifies primary copies in the local object store.
- Spill requests are sent to IO workers.
- IO workers write object values and metadata to external storage.
- The object directory is updated with the new locations of spilled objects.
- Primary copies are evicted from the object store.
- Once an object's reference count reaches zero, the owner notifies the raylet to remove it from external storage.

### Object Restoration
Spilled objects are restored on demand. The raylet either restores them from external storage or fetches them from another node. If a remote raylet has the object on local storage, it reads and sends it over the network.

### Efficiency Considerations
- Spilling many small objects individually is inefficient due to IO overhead. Ray fuses objects smaller than 100MB into a single file to mitigate this.
- Multi-directory spilling is supported, utilizing multiple file systems to enhance bandwidth and storage capacity.

### Known Limitations
- **Local Storage:** Spilled objects are lost if the node storing them fails. Ray attempts recovery as if they were lost from shared memory.
- **Owner Dependency:** A spilled object is unreachable if its owner is lost, as the owner maintains the object's location data.
- **Pinned Objects:** Objects in use by the application, such as those accessed via `ray.get`, are pinned and not spillable until released. Task arguments are pinned for the task's duration.

# Overview of scheduling strategies

Ray provides different scheduling strategies that you can set on your task.

We will go over:
- How a raylet assess feasibility and availability of nodes
- How every scheduling strategy/policy works and when you should use it

## Node classification

Given a resource requirement, a raylet classifies a node as one of the following:
- feasible
    - available
    - not available
- infeasible node 

Let's understand this by looking at an example task `my_task` that has a resource requirement of 3 CPUs:

- all nodes with >= 3 CPUs are classified as **feasible**
    - all **feasible nodes** that have >= 3 CPUs **idle** are classified as **available**

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/raylet_node_classification_v3.svg" width="1000px">

## Default scheduling strategy

This is the default scheduling policy used by Ray

### Motivation

Ray attempts to strike a balance between favoring nodes that already cater for data locality and favoring those that have low resource utilization.

### How does it work?
It is a hybrid policy that combines the following two heuristics:
- Bin packing heuristic
- Load balancing heuristic

<!-- ### References:
- See code here:
    - [Default Hybrid Scheduling Policy is defined here](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/policy/hybrid_scheduling_policy.cc) -->

The diagram below shows the policy in action in a bin-packing heuristic/mode

Note the **Local Node** shown in the diagram is the node that is local to the raylet that received the worker lease request - which in almost all cases is the raylet that satisfies data locality requirements.

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/scheduling_policy_hybrid_policy_binpacking_v4.svg" width="1200px">

The diagram below shows the policy in action in a load balancing heuristic. 

This occurs when our preferred local node is heavily being utilized. The strategy will now spread new tasks among other feasible and available nodes.

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/scheduling_policy_hybrid_policy_balancing_v4.svg" width="1200px">

## SPREAD scheduling strategy

### How does it work?
It behaves like a best-effort round-robin. It spreads across all the available nodes first and then the feasible nodes.

### Use-cases
- When you want to load-balance your tasks across nodes. e.g. you are building a web service and want to avoid overloading certain nodes.


<!-- ### References:
- See code here
    - [Spread Scheduling Policy is defined here](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/policy/spread_scheduling_policy.cc)
  -->

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/scheduling_policy_spread_v3.svg" width="800px">

### Sample code

In [ ]:
import ray


@ray.remote(scheduling_strategy="SPREAD")
def spread_default_func():
    return 2


ray.get(spread_default_func.remote())

## Placement Group Scheduling Strategy

In cases when we want to treat a set of resources as a single unit, we can use placement groups.

### How does it work?

- A **placement group** is formed from a set of **resource bundles**
  - A **resource bundle** is a list of resource requirements that fit in a single node
- A **placement group** can specify a **placement strategy** that determines how the **resource bundles** are placed
  - The **placement strategy** can be one of the following:
    - **PACK**: pack the **resource bundles** into as few nodes as possible
    - **SPREAD**: spread the **resource bundles** across as many nodes as possible
    - **STRICT_PACK**: pack the **resource bundles** into as few nodes as possible and fail if not possible
    - **STRICT_SPREAD**: spread the **resource bundles** across as many nodes as possible and fail if not possible
- **Placement Groups** are **atomic** 
  -  i.e. either all the **resource bundles** are placed or none are placed
  -  GCS uses a two-phase commit protocol to ensure atomicity

### Use-cases

Placement groups are used for **atomic gang scheduling**. Imagine the use case of a distributed training that requires 4 GPU nodes total. Other distributed schedulers might first reserve 3 GPUs and hang waiting for the fourth hogging resources in the meantime. Ray, instead, will either reserve all 4 GPUs or it will fail scheduling.

- Use SPREAD when you want to load-balance your tasks across nodes. e.g. you are building a web service and want to avoid overloading certain nodes.
- Use PACK when you want to maximize resource utilization. e.g. you are running training and want to cut costs by packing all your resource bundles on a small subset of nodes.

<!-- ### References
- [See code here for Bundle Scheduling Policy](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/policy/bundle_scheduling_policy.cc) -->

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/scheduling_policy_placement_group_v3.svg" width="600px">

### Example Code

In [ ]:
import ray
from ray.util.scheduling_strategies import PlacementGroupSchedulingStrategy
from ray.util.placement_group import (
    placement_group,
    placement_group_table,
    remove_placement_group,
)

# Reserve a placement group of 1 bundle that reserves 0.1 CPU
pg = placement_group([{"CPU": 0.1}], strategy="PACK", name="my_pg_2")

# Wait until placement group is created.
ray.get(pg.ready(), timeout=10)

# Inspect placement group states using the table
print(placement_group_table(pg))


@ray.remote(
    scheduling_strategy=PlacementGroupSchedulingStrategy(
        placement_group=pg,
    ),
    # task requirement needs to be less than placement group capacity
    num_cpus=0.1,
)
def placement_group_schedule():
    return 2


out = ray.get(placement_group_schedule.remote())
print(out)

# Remove placement group.
remove_placement_group(pg)

## Node affinity strategy

### How does it work?
It assigns a task to a given node in either a strict or soft manner.

### Use-cases
- When you want to ensure that your task runs on a specific node


<!-- ### References:
- See code here
    - [Node Affinity Policy is defined here](https://github.com/ray-project/ray/blob/releases/2.8.1/src/ray/raylet/scheduling/policy/node_affinity_scheduling_policy.cc)
  -->

<img src="https://assets-training.s3.us-west-2.amazonaws.com/ray-core/task-actor-lifecycle/v2/scheduling/scheduling_policy_node_affinity_v3.svg" width="1200px">

### Sample code

In [ ]:
import ray
from ray.util.scheduling_strategies import NodeAffinitySchedulingStrategy

# pin this task to only run on the current node id
run_on_same_node = NodeAffinitySchedulingStrategy(
    node_id=ray.get_runtime_context().get_node_id(), 
    soft=False,
)

@ray.remote(
    scheduling_strategy=run_on_same_node,
    num_cpus=0,
)
def node_affinity_schedule():
    return 2


ray.get(node_affinity_schedule.remote())